In [1]:
import numpy as np
from orchid import *

In [2]:
from eeroqlab.instruments import MagnetController, KoradKA3305P, BlueFors, Vulcan, QDac2Eeroq
from pymeasure.instruments.yokogawa.yokogawa7651 import Yokogawa7651
from pymeasure.instruments.srs.sr830 import SR830
from eeroqlab.instruments import E5071

In [3]:
vulcan  = Vulcan()
yoko    = Yokogawa7651('GPIB0::19::INSTR')
korad   = KoradKA3305P(resourceName='COM9', includeSCPI=False)
bf      = BlueFors("bf_fridge","B://", 1, 2, 3, 4, 5, 6, 1, 2, 3, 5, 6, 7, 8)
lockin  = SR830(name="lockin", adapter='GPIB0::8::INSTR')
qdac    = QDac2Eeroq(name="qdac", address="TCPIP::10.103.204.39::5025::SOCKET", timeout=60)
vna     = E5071(name="E5071C", address="e5071c.eeroq.local")

Connected to: QDevil QDAC-II (serial:86, firmware:13-1.57) in 0.05s


In [4]:
# gate names to qdac channel mapping
gates = {
    "local_in":    9,         # reservoir 1 of LOCAL ST 
    "local_out":   13,        # reservoir 2 of LOCAL ST 
    "local_gate":  11,        # central gate of LOCAL ST
    "local_top":   15,        # top filament electrode of LOCAL ST
    "remote_gate": 3,         # this is central gate of the REMOTE ST
    "remote_top":  17,        # top filament electrode of REMOTE ST
    "connect":     7,         # this is the channel connecting LOCAL and REMOTE devices
}
qdac.setup_gates(gates)
qdac.snapshot()

╭────────────┬─────────────┬──────────────┬─────────────┬───────────────┬──────────────┬───────────╮
│   local_in │   local_out │   local_gate │   local_top │   remote_gate │   remote_top │   connect │
├────────────┼─────────────┼──────────────┼─────────────┼───────────────┼──────────────┼───────────┤
│         -0 │          -0 │           -0 │           0 │             0 │            0 │         0 │
╰────────────┴─────────────┴──────────────┴─────────────┴───────────────┴──────────────┴───────────╯


In [5]:
ctx = ExperimentContext(
    data_root="./example_data",
    metadata={
        "sample": "chip_A1",
        "operator": "Niyaz",
        "fridge": "BlueFors_1",
        "cooldown": 42,
    },
)

In [6]:
ctx.add_instrument("lockin", lockin, backend="pymeasure")
ctx.add_instrument("qdac", qdac, backend="qcodes")
ctx.add_instrument("vna", vna, backend="custom")
ctx.add_instrument("vulcan", vulcan, backend="custom")
ctx.add_instrument("bluefors", bf, backend="qcodes")

InstrumentAdapter('bluefors', backend='qcodes')

In [7]:
ctx.add_parameter("T6", get_func=lambda: bf.get_temperature(6), unit="K")
ctx.add_parameter("Vch", get_func=lambda: qdac.local_gate.dc_constant_V(), set_func=lambda x: qdac.local_gate.ramp_V(x), unit="V")

def set_Vr(x):
    qdac.local_in.ramp_V(x)
    qdac.local_out.ramp_V(x)

ctx.add_parameter("Vr", get_func=qdac.local_in.dc_constant_V, set_func=set_Vr, unit="V")
ctx.add_parameter("fac", instrument="lockin", attr="frequency", unit="Hz")
ctx.add_parameter("Vac", instrument="lockin", attr="sine_voltage", unit="Vrms")
ctx.add_parameter("tau_flash", get_func=lambda: vulcan.get_duration(), set_func=lambda x: vulcan.set_duration(x), unit="s")

Parameter('tau_flash', callable, unit='s')

In [13]:
ctx.snapshot()

Name       Type    Value        Unit
---------  ------  -----------  -------------------
T6         param   0.023321     K
Vch        param   -0.0         V
Vr         param   -0.0         V
fac        param   34730.0      Hz
Vac        param   1.0          Vrms
tau_flash  param   100          s
Vx         scalar  1.01329e-06  V
Vy         scalar  5.90089e-06  V
S21        image   [[...]]      ['Hz', 'dB', 'deg']


In [9]:
ctx.add_readout("Vx", kind="scalar", get_func=lambda: lockin.x, unit="V")
ctx.add_readout("Vy", kind="scalar", get_func=lambda: lockin.y, unit="V")
ctx.add_readout("S21", kind="image", shape=(1601,3),
    get_func=vna.take_one_averaged_trace,
    unit=["Hz","dB","deg"],
    contains=["f","mag","phase"],)

Readout('S21', image, shape=(1601, 3))

In [12]:
ctx["fac"] = 34.73e3

In [ ]:
@dataclass
class VNAconfig():
    name: str
    measure: str
    f1: float
    f2: float
    power: float
    ifbw: float

def configure_vna(config: VNAconfig):
    vna.set_measure(config.measure)
    vna.set_power(config.power)
    vna.set_start_frequency(config.f1)
    vna.set_stop_frequency(config.f2)
    vna.set_ifbw(config.ifbw)

In [16]:
qdac.close()
# korad.close()